In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
people_path = train_dir / "la_mmh_training_person_dim.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [lms_path, video_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [2]:
hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

phantom_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("phantom"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Loaded {len(phantom_answers):,} unique Phantom answers")
display(phantom_answers.head())


Loaded 62 unique Phantom answers


,P_ID
0,P00000573
1,P00000693
2,P00000700
3,P00000987
4,P00001206


In [3]:
# Find repeated activity details while excluding each row's primary key.
lms_columns = ["PIT_ID", "P_ID", "DATE", "TIMESTAMP", "LMS_ITEM_ID", "LOGIN_ID", "INTERACTION_COUNT", "TOTAL_SECONDS_BEFORE_NEXT_INTERACTION"]
video_columns = ["PVT_ID", "P_ID", "DATE", "TIMESTAMP", "VIDEO_ID", "TOTAL_MINUTES_DELIVERED", "VIEWING_TYPE"]
phantom_ids = set(phantom_answers["P_ID"])


In [4]:
def duplicate_evidence(path, columns, row_id, source, target_ids, chunksize=200_000):
    parts = []
    compare_columns = [column for column in columns if column != row_id]

    for chunk in pd.read_csv(path, usecols=columns, dtype="string", chunksize=chunksize):
        chunk = chunk.loc[chunk["P_ID"].isin(target_ids)].copy()
        if not chunk.empty:
            parts.append(chunk)

    if not parts:
        return pd.DataFrame(columns=columns + ["source", "duplicate_count"])

    activity = pd.concat(parts, ignore_index=True)
    duplicated = activity.loc[activity.duplicated(subset=compare_columns, keep=False)].copy()
    if duplicated.empty:
        return pd.DataFrame(columns=columns + ["source", "duplicate_count"])
    duplicated["duplicate_count"] = duplicated.groupby(compare_columns, dropna=False)[row_id].transform("size")
    return duplicated.assign(source=source).drop_duplicates()

phantom_evidence = pd.concat([
    duplicate_evidence(lms_path, lms_columns, "PIT_ID", "LMS", phantom_ids),
    duplicate_evidence(video_path, video_columns, "PVT_ID", "Video", phantom_ids),
], ignore_index=True, sort=False)
phantom_evidence = phantom_evidence.sort_values("P_ID").reset_index(drop=True)
print(f"Official answers with duplicate-row evidence: {phantom_evidence['P_ID'].nunique():,} of {len(phantom_ids):,}")
display(phantom_evidence.head(20))


Official answers with duplicate-row evidence: 62 of 62


,PIT_ID,DATE,P_ID,LMS_ITEM_ID,TIMESTAMP,LOGIN_ID,INTERACTION_COUNT,TOTAL_SECONDS_BEFORE_NEXT_INTERACTION,duplicate_count,source,PVT_ID,VIDEO_ID,TOTAL_MINUTES_DELIVERED,VIEWING_TYPE
0,<NA>,2024-02-06,P00000573,<NA>,2024-02-06T22:26:31.000Z,<NA>,<NA>,<NA>,2,Video,PVT0001276275,V000004421,0.6854,Interactive Desktop On Demand
1,<NA>,2024-02-03,P00000573,<NA>,2024-02-03T16:13:01.000Z,<NA>,<NA>,<NA>,2,Video,PVT0001063093,V000019824,19.4079,Interactive Desktop On Demand
2,<NA>,2024-09-02,P00000573,<NA>,2024-09-02T21:26:38.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000269169,V000001755,0.1345,Interactive Desktop On Demand
3,<NA>,2024-09-02,P00000573,<NA>,2024-09-02T16:10:17.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000770363,V000008620,0.138,Interactive Desktop On Demand
4,<NA>,2024-08-31,P00000573,<NA>,2024-08-31T11:25:04.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000857149,V000008620,0.1602,Interactive Desktop On Demand
5,<NA>,2024-04-15,P00000573,<NA>,2024-04-15T06:58:31.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000972216,V000013065,0.071,Interactive Desktop On Demand
6,<NA>,2024-08-31,P00000573,<NA>,2024-08-31T11:31:05.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000891455,V000008620,0.1036,Interactive Desktop On Demand
7,<NA>,2024-08-31,P00000573,<NA>,2024-08-31T11:21:03.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000509503,V000009526,3.887,Interactive Desktop On Demand
8,<NA>,2023-12-27,P00000573,<NA>,2023-12-27T12:45:18.000Z,<NA>,<NA>,<NA>,2,Video,PVT0000974552,V000024633,62.8812,Interactive Desktop On Demand
9,<NA>,2023-12-26,P00000573,<NA>,2023-12-26T21:52:40.000Z,<NA>,<NA>,<NA>,2,Video,PVT0001422713,V000024633,0.4565,Interactive Desktop On Demand


In [5]:
phantom_submission = phantom_answers.copy()
phantom_submission.insert(0, "Monster Type", "Phantom")

print(f"Prepared {len(phantom_submission):,} unique Phantom IDs")
display(phantom_submission)


Prepared 62 unique Phantom IDs


,Monster Type,P_ID
0,Phantom,P00000573
1,Phantom,P00000693
2,Phantom,P00000700
3,Phantom,P00000987
4,Phantom,P00001206
...,...,...
57,Phantom,P00019270
58,Phantom,P00020479
59,Phantom,P00021320
60,Phantom,P00021563
